<a href="https://colab.research.google.com/github/nkefeyan-22-26/ECON5200-Applied-Data-Analytics-in-Economics/blob/main/Lab13/Lab%2013%3A%20The%20Architecture%20of%20Dimensionality.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
import pandas as pd
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

# Step 1: Ingestion and Naive Model
url = "https://raw.githubusercontent.com/nkefeyan-22-26/ECON5200-Applied-Data-Analytics-in-Economics/refs/heads/main/Data/Zillow_California_2026_Hedonic.csv"
df = pd.read_csv(url)
df

,Property_Age,Distance_to_Tech_Hub,Sale_Price
0,77.5,38.1,684100.56
1,11.0,95.1,413634.22
2,47.7,73.5,456709.35
3,61.9,60.3,624533.95
4,100.8,16.4,870137.54
...,...,...,...
995,87.7,10.1,932592.35
996,21.2,91.8,412741.12
997,96.5,14.5,880901.56
998,20.1,95.1,396659.79


In [15]:
naive_model = smf.ols('Sale_Price ~ Property_Age', data=df).fit()
print(naive_model.summary())
print("\nNaive Age Coefficient:", naive_model.params['Property_Age'])

                            OLS Regression Results                            
Dep. Variable:             Sale_Price   R-squared:                       0.757
Model:                            OLS   Adj. R-squared:                  0.757
Method:                 Least Squares   F-statistic:                     3105.
Date:                Fri, 13 Mar 2026   Prob (F-statistic):          1.26e-308
Time:                        18:37:04   Log-Likelihood:                -12818.
No. Observations:                1000   AIC:                         2.564e+04
Df Residuals:                     998   BIC:                         2.565e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept     3.013e+05   7218.570     41.742   

In [23]:
# Step 2: The Multivariate Model
multi_model = smf.ols('Sale_Price ~ Property_Age + Distance_to_Tech_Hub', data=df).fit()
print(multi_model.summary())
print("\nMultivariate Age Coefficient:", multi_model.params['Property_Age'])

                            OLS Regression Results                            
Dep. Variable:             Sale_Price   R-squared:                       0.954
Model:                            OLS   Adj. R-squared:                  0.954
Method:                 Least Squares   F-statistic:                 1.040e+04
Date:                Fri, 13 Mar 2026   Prob (F-statistic):               0.00
Time:                        18:40:02   Log-Likelihood:                -11982.
No. Observations:                1000   AIC:                         2.397e+04
Df Residuals:                     997   BIC:                         2.399e+04
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept             1.203e+06 

In [16]:
# Step 3: FWL Theorem Manual Proof
# 3a: Partial out distance from Price
res_y_model = smf.ols('Sale_Price ~ Distance_to_Tech_Hub', data=df).fit()
df['Price_Residuals'] = res_y_model.resid

In [17]:
# 3b: Partial out distance from Age
res_x_model = smf.ols('Property_Age ~ Distance_to_Tech_Hub', data=df).fit()
df['Age_Residuals'] = res_x_model.resid

In [18]:
# 3c: Regress Residuals on Residuals (-1 removes the intercept for exact mathematical matching)
fwl_model = smf.ols('Price_Residuals ~ Age_Residuals - 1', data=df).fit()
print("\nFWL Isolated Age Coefficient:", fwl_model.params['Age_Residuals'])


FWL Isolated Age Coefficient: -2063.129216802139


In [24]:
"""
Multivariate OLS Regression — 3D Scatter + Fitted Hyperplane
=====================================================================
Predicting Sale_Price ~ Property_Age + Distance_to_Tech_Hub
Libraries: numpy, pandas, statsmodels, plotly.graph_objects
"""

import numpy as np
import pandas as pd
import statsmodels.api as sm
import plotly.graph_objects as go

# ── 0. REPRODUCIBLE SAMPLE DATA ──────────────────────────────────────────────
# Replace this block with your own DataFrame if you already have one.
np.random.seed(42)
n = 200

Property_Age        = np.random.uniform(0, 40, n)          # years
Distance_to_Tech_Hub = np.random.uniform(1, 50, n)         # km

# True data-generating process (DGP) + noise
Sale_Price = (
    500_000
    - 3_500  * Property_Age
    - 4_200  * Distance_to_Tech_Hub
    + np.random.normal(0, 15_000, n)                        # residual noise
)

df = pd.DataFrame({
    "Sale_Price":           Sale_Price,
    "Property_Age":         Property_Age,
    "Distance_to_Tech_Hub": Distance_to_Tech_Hub,
})

# ── 1. FIT OLS MODEL ─────────────────────────────────────────────────────────
X = sm.add_constant(df[["Property_Age", "Distance_to_Tech_Hub"]])  # adds intercept column
y = df["Sale_Price"]

model  = sm.OLS(y, X).fit()
print(model.summary())

# ── 2. EXTRACT COEFFICIENTS ───────────────────────────────────────────────────
# model.params is a pandas Series indexed by variable name.
# Accessing by name is safer than by position — immune to column reordering.
intercept  = model.params["const"]                  # β₀
coef_age   = model.params["Property_Age"]           # β₁
coef_dist  = model.params["Distance_to_Tech_Hub"]   # β₂

print(f"\nβ₀ (intercept)          : {intercept:,.2f}")
print(f"β₁ (Property_Age)       : {coef_age:,.2f}")
print(f"β₂ (Distance_to_Tech_Hub): {coef_dist:,.2f}")

# ── 3. BUILD THE MESHGRID FOR THE REGRESSION SURFACE ─────────────────────────
# We need a 2-D grid over the predictor space so Plotly can draw a smooth
# surface.  np.linspace creates evenly-spaced tick values across each axis;
# np.meshgrid then combines them into two 2-D arrays (one per axis) where
# every (i, j) cell represents one (Age, Distance) coordinate pair.

age_range  = np.linspace(df["Property_Age"].min(),
                         df["Property_Age"].max(), 60)          # 60 ticks on X
dist_range = np.linspace(df["Distance_to_Tech_Hub"].min(),
                         df["Distance_to_Tech_Hub"].max(), 60)  # 60 ticks on Y

# age_grid and dist_grid are both (60 × 60) matrices.
# Each row of age_grid  holds one fixed age repeated across all distances.
# Each col of dist_grid holds one fixed distance repeated across all ages.
age_grid, dist_grid = np.meshgrid(age_range, dist_range)

# Apply the OLS equation element-wise across the entire grid:
#   Ŷ = β₀ + β₁·Age + β₂·Distance
# Because age_grid and dist_grid are 2-D arrays, price_grid is also (60 × 60).
# Each cell is the model's predicted Sale_Price at that (Age, Distance) combo.
price_grid = intercept + coef_age * age_grid + coef_dist * dist_grid

# ── 4. BUILD PLOTLY FIGURE ────────────────────────────────────────────────────
fig = go.Figure()

# — 4a. Actual data points (scatter) ————————————————————————————————————————
fig.add_trace(go.Scatter3d(
    x=df["Property_Age"],
    y=df["Distance_to_Tech_Hub"],
    z=df["Sale_Price"],
    mode="markers",
    marker=dict(
        size=4,
        color=df["Sale_Price"],        # colour by price for extra dimension
        colorscale="Viridis",
        opacity=0.75,
        colorbar=dict(title="Sale Price ($)", x=1.02),
        showscale=True,
    ),
    name="Observed Data",
    hovertemplate=(
        "<b>Observed</b><br>"
        "Property Age: %{x:.1f} yrs<br>"
        "Distance: %{y:.1f} km<br>"
        "Sale Price: $%{z:,.0f}<extra></extra>"
    ),
))

# — 4b. Fitted regression hyperplane (surface) ————————————————————————————
fig.add_trace(go.Surface(
    x=age_grid,          # (60×60) grid of Property_Age values
    y=dist_grid,         # (60×60) grid of Distance_to_Tech_Hub values
    z=price_grid,        # (60×60) grid of OLS-predicted Sale_Price values
    colorscale="RdBu",
    opacity=0.55,
    showscale=False,
    name="OLS Hyperplane",
    hovertemplate=(
        "<b>OLS Prediction</b><br>"
        "Property Age: %{x:.1f} yrs<br>"
        "Distance: %{y:.1f} km<br>"
        "Predicted Price: $%{z:,.0f}<extra></extra>"
    ),
))

# — 4c. Layout ——————————————————————————————————————————————————————————————
r2   = model.rsquared
adj_r2 = model.rsquared_adj

fig.update_layout(
    title=dict(
        text=(
            f"OLS Regression: Sale Price ~ Property Age + Distance to Tech Hub<br>"
            f"<sup>R² = {r2:.4f} | Adj. R² = {adj_r2:.4f} | "
            f"β₁(Age) = {coef_age:,.0f} | β₂(Dist) = {coef_dist:,.0f}</sup>"
        ),
        x=0.5,
    ),
    scene=dict(
        xaxis_title="Property Age (years)",
        yaxis_title="Distance to Tech Hub (km)",
        zaxis_title="Sale Price ($)",
        xaxis=dict(backgroundcolor="rgb(240,240,255)"),
        yaxis=dict(backgroundcolor="rgb(240,255,240)"),
        zaxis=dict(backgroundcolor="rgb(255,245,240)"),
        camera=dict(eye=dict(x=1.6, y=-1.6, z=0.9)),  # default viewing angle
    ),
    legend=dict(x=0.01, y=0.95),
    margin=dict(l=0, r=0, t=80, b=0),
    height=720,
)

fig.show()
# fig.write_html("ols_3d_regression.html")   # ← uncomment to save as HTML

                            OLS Regression Results                            
Dep. Variable:             Sale_Price   R-squared:                       0.960
Model:                            OLS   Adj. R-squared:                  0.960
Method:                 Least Squares   F-statistic:                     2384.
Date:                Fri, 13 Mar 2026   Prob (F-statistic):          9.16e-139
Time:                        18:41:04   Log-Likelihood:                -2203.7
No. Observations:                 200   AIC:                             4413.
Df Residuals:                     197   BIC:                             4423.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                 5.013e+05 